# Module 5 — Hyperparameter Tuning

**By the end of this notebook, you will be able to:**
- Use `GridSearchCV` together with `GroupKFold` — the ready-made scikit-learn version of the cross-validation you built by hand in Module 3
- Explain why an untuned, more complex model can be *worse* than a simple one
- Read a hyperparameter search's results, not just its single "best" answer

**Context:** Module 4 left you with 8 RFE-selected columns and a `LinearRegression` baseline around RMSE 27.6. This module asks whether a more flexible model — `XGBRegressor` — can do better, and what happens if you use it without tuning it at all. `uv sync --group ml` is required for this module (`xgboost`).

## Rebuild the Module 4 pipeline

**Exercise:** Same sequence as Module 4 (`restrict_to_scope` → `columns_above_missing_threshold` → `drop_columns` → `fill_missing_by_city` → `add_temporal_features`), then reuse `selection.select_features_rfe` to get back the same 8 columns.

In [5]:
# Rebuild the cleaned four-city dataset and the RFE-selected feature set
from pathlib import Path
import sys

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src" / "air_quality").is_dir():
        sys.path.insert(0, str(candidate / "src"))
        break

from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from air_quality import data, evaluation, features, selection, tuning

ALL_CITIES = ["Kampala", "Nairobi", "Lagos", "Bujumbura"]
train_df, _ = data.load_datasets()
all_columns = train_df.columns.tolist()
scoped = data.restrict_to_scope(train_df, ALL_CITIES, all_columns)
columns_to_drop = data.columns_above_missing_threshold(scoped, threshold=0.7)
cleaned = data.drop_columns(scoped, columns_to_drop)
fillable_columns = [c for c in cleaned.columns if c not in ("city", "date")]
cleaned = data.fill_missing_by_city(cleaned, fillable_columns)
enriched = features.add_temporal_features(cleaned)
feature_cols = features.feature_columns(enriched)
rfe_selected = selection.select_features_rfe(
    LinearRegression(), enriched, feature_cols, target_col="pm2_5",
    n_features_to_select=8,
)
print("RFE-selected features:", rfe_selected)


RFE-selected features: ['sulphurdioxide_so2_column_number_density', 'sulphurdioxide_so2_slant_column_number_density', 'carbonmonoxide_co_column_number_density', 'nitrogendioxide_no2_column_number_density', 'nitrogendioxide_tropospheric_no2_column_number_density', 'nitrogendioxide_stratospheric_no2_column_number_density', 'nitrogendioxide_no2_slant_column_number_density', 'formaldehyde_tropospheric_hcho_column_number_density']


## What if you don't tune at all?

Try `XGBRegressor` with its defaults — no hyperparameters chosen deliberately — and evaluate it exactly like every other feature set this session, with `evaluate_group_cv`.

In [6]:
# Given: an untuned model, evaluated the same way as everything else this session
result_untuned = evaluation.evaluate_group_cv(XGBRegressor(random_state=42), enriched, rfe_selected, groups_col="city")
{k: v for k, v in result_untuned.items() if k != "folds"}

{'rmse_mean': np.float64(31.880109856504554),
 'rmse_std': np.float64(12.193911186038605),
 'mae_mean': np.float64(16.70905422442226),
 'mae_std': np.float64(6.12432244982878),
 'r2_mean': np.float64(-0.7731593197523958),
 'r2_std': np.float64(1.4275786489341122)}

**Worse than Module 4's plain `LinearRegression` (RMSE ≈ 27.6) on the exact same 8 columns.** A more flexible model is not automatically better — without tuning, `XGBRegressor` has no reason to fit this particular, fairly small, fairly linear dataset well. Flexibility has to be pointed in the right direction by its hyperparameters, or it can hurt more than it helps.

[`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) does that search for you — and, combined with [`GroupKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html) as its `cv`, it is the exact guarantee you built by hand in Module 3 (a group never appears in both train and validation for a fold), just applied automatically across every hyperparameter combination instead of manually across every city.

## Tune it properly

Open `src/air_quality/tuning.py` (new file, already scaffolded) and complete `tune_xgboost`. Its docstring and `tests/test_tuning.py` specify exactly what it should do. Run `uv run pytest tests/test_tuning.py -v` until it passes, then apply it here with a small grid — `{"max_depth": [3, 5], "learning_rate": [0.05, 0.2]}`, 4 combinations, nothing exhaustive.

In [7]:
tuning_result = tuning.tune_xgboost(
    enriched,
    feature_cols=rfe_selected,
    param_grid={"max_depth": [3, 5], "learning_rate": [0.05, 0.2]},
    target_col="pm2_5",
    groups_col="city",
)
print("Best parameters:", tuning_result["best_params"])
print(f"Best mean RMSE: {tuning_result['best_rmse']:.2f}")


Best parameters: {'learning_rate': 0.05, 'max_depth': 3}
Best mean RMSE: 27.57


In [8]:
# Given: every combination tried, not just the best one
import pandas as pd

pd.DataFrame(tuning_result["cv_results"]).sort_values("rmse")

,params,rmse,mae,r2
0,"{'learning_rate': 0.05, 'max_depth': 3}",27.567362,14.244063,0.084508
2,"{'learning_rate': 0.2, 'max_depth': 3}",28.787906,15.894646,-0.073808
1,"{'learning_rate': 0.05, 'max_depth': 5}",28.890075,14.765108,-0.092112
3,"{'learning_rate': 0.2, 'max_depth': 5}",31.071349,17.276857,-0.491979


Notice the table has `rmse`, `mae` and `r2` for every combination — the same three metrics `regression_metrics()` has always reported together. `GridSearchCV` still ranks combinations by `rmse` alone (`refit="rmse"` in `tune_xgboost`) since a search needs one number to compare against, but nothing stops you from reading the other two for the combination it picks.

**Continue on the site:** [Module 5 — Hyperparameter Tuning](https://hub.imt-atlantique.fr/datascience-toolkit/session3/practical_5/)